# 01 · North Carolina Flood-Event EDA

**StormRoute — Carolina Data Challenge 2026, Natural Science Track**

This notebook is the **Milestone 2 verification gate**. No production modeling,
scoring, API, or website work begins until it has been executed end to end,
reviewed by two teammates, and merged.

It must read as an analytical narrative, not a scratchpad: each section states the
question it tests, answers it with code, and closes with a short conclusion.

| | |
|---|---|
| **Run mode** | `STORMROUTE_DATA_MODE=sample` in CI (tracked fixtures) · `full` in Codespaces |
| **Execute** | `make eda` — restart kernel, run all, fail on any exception |
| **Figures** | exported to `outputs/figures/eda_*.png` with stable names |
| **Report** | findings summarized in `reports/eda/eda_findings.md` |

> **Status:** sections 1–5 and 9–11 implemented. Sections 6–8 (Econ/Stats) and 12 pending.

## 1. Objective and scope

**Question.** What exactly are we predicting, and what are we deliberately not predicting?

A traveler decides on a route and a departure time before the trip starts. StormRoute
estimates, for every North Carolina county and six-hour window, the probability that a
reported **Flood, Flash Flood, or Debris Flow** event begins there. Route-level exposure is
built from those probabilities later.

The definitions below are read from `configs/`, not typed here, so this notebook cannot
drift away from the pipeline.

In [ ]:
import pandas as pd

from stormroute.config import load_config

DATA = load_config("data")
MODEL = load_config("model")

splits = {name: spec["years"] for name, spec in MODEL["splits"].items() if isinstance(spec, dict)}
pd.DataFrame(
    [
        (
            "Geography",
            f"{DATA['geography']['state']} (state FIPS {DATA['geography']['state_fips']}), "
            f"{DATA['geography']['expected_county_count']} counties",
        ),
        ("Period", f"{DATA['period']['start_year']}–{DATA['period']['end_year']}"),
        (
            "Prediction unit",
            f"county × {DATA['prediction_unit']['window_hours']}-hour window, "
            f"anchored at {DATA['prediction_unit']['window_anchor_utc']} UTC",
        ),
        ("Qualifying hazards", ", ".join(DATA["label"]["hazard_event_types"])),
        ("Positive label", DATA["label"]["onset_rule"]),
        ("Negative label means", DATA["label"]["negative_meaning"]),
        *((f"Split: {name}", ", ".join(map(str, years))) for name, years in splits.items()),
        ("Never a model input", ", ".join(MODEL["features"]["excluded"])),
    ],
    columns=["Definition", "Value"],
).set_index("Definition")

**Conclusion.** The scope is fixed by configuration. Injuries, deaths, and damage are
**descriptive outcomes only**: they are recorded after an event and can never be model
inputs. Sections 3–5 test whether the NOAA records support this scope, and section 9 tests
whether the label can be built.

## 2. Reproducibility information

**Question.** Could a teammate re-run this notebook tomorrow and get the same numbers?

`STORMROUTE_DATA_MODE=sample` runs on the tracked fixture (what CI does). `full` reads the
downloaded NOAA files named in the download record. Figures from a sample run go to
`outputs/figures/sample/`, which is ignored, so they can never overwrite the real ones.

In [ ]:
import hashlib
import json
import platform
import subprocess
import sys
from datetime import UTC, datetime
from importlib.metadata import version

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Markdown, display
from matplotlib.patches import Patch

from stormroute.config import REPO_ROOT, data_mode, data_path
from stormroute.data.noaa import (
    RAW_COLUMNS,
    RECORD_FILENAME,
    active_detail_files,
    clean_events,
    read_details,
)

MODE = data_mode()
SEED = MODEL["final_model"]["random_seed"]
FIGURES = (
    REPO_ROOT / "outputs" / "figures"
    if MODE == "full"
    else REPO_ROOT / "outputs" / "figures" / "sample"
)

if MODE == "sample":
    SOURCE_PATHS = [data_path("sample") / "storm_events_sample.csv"]
    sources = pd.DataFrame(
        {
            "file": [p.name for p in SOURCE_PATHS],
            "sha256": [hashlib.sha256(p.read_bytes()).hexdigest()[:16] for p in SOURCE_PATHS],
        }
    )
else:
    period = range(DATA["period"]["start_year"], DATA["period"]["end_year"] + 1)
    SOURCE_PATHS = active_detail_files(data_path("noaa_raw"), period)
    record = json.loads((data_path("noaa_raw") / RECORD_FILENAME).read_text())
    sources = pd.DataFrame(
        [
            {
                "year": record["files"][p.name]["year"],
                "file": p.name,
                "sha256": record["files"][p.name]["sha256"][:16],
                "recorded_at_utc": record["files"][p.name]["recorded_at_utc"],
            }
            for p in SOURCE_PATHS
        ]
    )

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    commit = "unavailable"

manifest = yaml.safe_load((REPO_ROOT / "data" / "data_manifest.yaml").read_text())
print(f"Analysis run (UTC): {datetime.now(UTC):%Y-%m-%d %H:%M}")
print(f"Data mode:          {MODE}")
print(f"Git commit:         {commit}")
print(
    f"Data manifest:      version {manifest['manifest_version']}, "
    f"updated {manifest['last_updated']}"
)
print(f"Python:             {sys.version.split()[0]} on {platform.system()}")
print(
    "Packages:           "
    + ", ".join(
        f"{pkg} {version(pkg)}"
        for pkg in ("stormroute", "pandas", "numpy", "matplotlib", "pyarrow")
    )
)
print(f"Random seed:        {SEED}")
print(f"Figures written to: {FIGURES.relative_to(REPO_ROOT)}")
sources

In [ ]:
# Figure standard: validated palette, recessive chrome, source note on every figure.
COLORS = {
    "surface": "#fcfcfb",
    "ink": "#0b0b0b",
    "ink_secondary": "#52514e",
    "muted": "#898781",
    "grid": "#e1e0d9",
    "baseline": "#c3c2b7",
    "accent": "#2a78d6",  # categorical slot 1: the thing the figure is about
    "context": "#898781",  # neutral for context marks
}
mpl.rcParams.update(
    {
        "figure.facecolor": COLORS["surface"],
        "axes.facecolor": COLORS["surface"],
        "savefig.facecolor": COLORS["surface"],
        "font.family": "sans-serif",
        "font.size": 11,
        "text.color": COLORS["ink"],
        "axes.labelcolor": COLORS["ink_secondary"],
        "xtick.color": COLORS["muted"],
        "ytick.color": COLORS["ink_secondary"],
        "axes.edgecolor": COLORS["baseline"],
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "grid.color": COLORS["grid"],
        "grid.linewidth": 0.8,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
    }
)
SOURCE_NOTE = "Source: NOAA NCEI Storm Events Database, details files " + (
    "(tracked sample fixture)"
    if MODE == "sample"
    else f"{DATA['period']['start_year']}–{DATA['period']['end_year']}"
)


def save_figure(fig: plt.Figure, name: str) -> None:
    FIGURES.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES / f"{name}.png", dpi=200, bbox_inches="tight")
    print(f"saved {(FIGURES / f'{name}.png').relative_to(REPO_ROOT)}")


def conclude(text: str) -> None:
    display(Markdown(f"**Conclusion.** {text}"))


def n(count: int, noun: str, plural: str | None = None) -> str:
    return f"{count:,} {noun if count == 1 else plural or noun + 's'}"


def pct(part: float, whole: float) -> str:
    if not whole or pd.isna(part):
        return "n/a"
    value = 100 * part / whole
    return f"{value:.3f}%" if 0 < value < 1 else f"{value:.1f}%"

**Conclusion.** Every number below traces to the files and checksums listed above, the
commit printed with them, and a fixed seed. Paths resolve from the repository root.

## 3. Schema and first-pass inspection

**Question.** Is the raw data shaped the way the NCEI documentation says it is?

Everything is read as **text**, exactly as published, so no county code or time loses a
leading zero to number parsing. The analysis reads a subset of the published columns.
Narratives are skipped because they are large and are never a model input.

In [ ]:
published_columns = pd.read_csv(SOURCE_PATHS[0], nrows=0).columns.tolist()
ANALYSIS_COLUMNS = [*RAW_COLUMNS, "BEGIN_DATE_TIME", "END_DATE_TIME", "BEGIN_LAT", "BEGIN_LON"]
raw = read_details(SOURCE_PATHS, ANALYSIS_COLUMNS)
raw_nc = raw[raw["STATE_FIPS"].str.zfill(2) == DATA["geography"]["state_fips"]]

print(f"Published columns: {len(published_columns)}  |  analysed: {len(ANALYSIS_COLUMNS)}")
print(f"Rows, all states:  {len(raw):,}  |  North Carolina: {len(raw_nc):,}")
print(f"Column dtypes:     {raw.dtypes.astype(str).value_counts().to_dict()} (all text by design)")

In [ ]:
raw_nc[
    [
        "EVENT_ID",
        "EVENT_TYPE",
        "CZ_TYPE",
        "CZ_FIPS",
        "CZ_NAME",
        "BEGIN_DATE_TIME",
        "END_DATE_TIME",
        "CZ_TIMEZONE",
        "DAMAGE_PROPERTY",
    ]
].sample(min(5, len(raw_nc)), random_state=SEED)

In [ ]:
duplicate_ids = int(raw["EVENT_ID"].duplicated().sum())
exact_duplicates = int(raw.duplicated().sum())
pd.DataFrame(
    [
        ("Duplicated EVENT_ID values", duplicate_ids),
        ("...of which exact duplicate rows", exact_duplicates),
        ("Distinct STATE values", raw["STATE"].nunique()),
        ("Distinct EVENT_TYPE values (all states)", raw["EVENT_TYPE"].nunique()),
        ("Distinct EVENT_TYPE values (NC)", raw_nc["EVENT_TYPE"].nunique()),
    ],
    columns=["Check", "Value"],
).set_index("Check")

In [ ]:
pd.crosstab(raw_nc["CZ_TYPE"], "NC rows").rename(
    index={"C": "C: county", "Z": "Z: forecast zone", "M": "M: marine"}
)

### Missingness (owner: Econ/Stats major)

Missingness table and `eda_missingness.png`, per the section split proposed in issue #5.

In [ ]:
if duplicate_ids == 0:
    identity = "every EVENT_ID is unique"
elif duplicate_ids == exact_duplicates:
    identity = (
        f"{n(duplicate_ids, 'EVENT_ID')} repeat, and every repeat is an exact duplicate row, "
        "which ingestion removes"
    )
else:
    identity = (
        f"{n(duplicate_ids - exact_duplicates, 'EVENT_ID')} repeat with **different** content, "
        "which ingestion rejects"
    )
conclude(
    f"Across {n(len(SOURCE_PATHS), 'source file')} there are {len(raw):,} rows, "
    f"{len(raw_nc):,} of them in North Carolina, and {identity}. NCEI publishes county "
    "codes, days, and times **unpadded** (e.g. `21` for Buncombe), so ingestion pads every "
    "one explicitly."
)

## 4. Time-field validation

**Question.** Can every event be placed on one unambiguous UTC timeline?

**Conversion rule.** NCEI splits each timestamp into `YEARMONTH`, `DAY`, and `TIME`, all
unpadded, and records a fixed offset in `CZ_TIMEZONE` (e.g. `EST-5`). Ingestion applies that
offset as written, with no daylight-saving lookup: `UTC = local − offset`.

The timestamps are then checked against NCEI's own `BEGIN_DATE_TIME` and `END_DATE_TIME`
text fields, an independent rendering of the same instant.

In [ ]:
events, report = clean_events(raw, source_files=[p.name for p in SOURCE_PATHS])
pd.DataFrame(
    [
        ("Events after all filters", report.events_out),
        ("Excluded: begin time invalid", report.excluded_invalid_begin),
        ("Excluded: end time invalid", report.excluded_invalid_end),
        ("Excluded: end before begin", report.excluded_end_before_begin),
        ("Kept: end time missing (labels need only begin)", report.missing_end_kept),
        ("Kept: zero duration (end = begin)", report.zero_duration),
    ],
    columns=["Timestamp check", "Events"],
).set_index("Timestamp check")

In [ ]:
published = (
    raw_nc.drop_duplicates("EVENT_ID")
    .set_index("EVENT_ID")
    .loc[events["event_id"], ["BEGIN_DATE_TIME", "END_DATE_TIME", "CZ_TIMEZONE"]]
)
offset = pd.to_timedelta(published["CZ_TIMEZONE"].str.extract(r"-(\d+)$")[0].astype(int), unit="h")


def as_local(stamps: pd.Series) -> pd.Series:
    return pd.to_datetime(stamps.replace("", None), format="%d-%b-%y %H:%M:%S")


begin_local = as_local(published["BEGIN_DATE_TIME"]).to_numpy()
end_local = as_local(published["END_DATE_TIME"]).to_numpy()
ours_begin = (events["begin_utc"].dt.tz_localize(None) - offset.to_numpy()).to_numpy()
ours_end = (events["end_utc"].dt.tz_localize(None) - offset.to_numpy()).to_numpy()

begin_mismatch = int((begin_local != ours_begin).sum())
end_both = ~pd.isna(end_local) & ~pd.isna(ours_end)
end_mismatch = int((end_local[end_both] != ours_end[end_both]).sum())
print(f"Begin times disagreeing with BEGIN_DATE_TIME: {begin_mismatch} of {len(events):,}")
print(f"End times disagreeing with END_DATE_TIME:     {end_mismatch} of {int(end_both.sum()):,}")
print(f"CZ_TIMEZONE values on NC rows: {raw_nc['CZ_TIMEZONE'].value_counts().to_dict()}")

In [ ]:
local_begin = pd.Series(begin_local, index=events.index)
local_end = pd.Series(end_local, index=events.index)
on_boundary = (events["begin_utc"].dt.minute == 0) & (events["begin_utc"].dt.hour % 6 == 0)
CASES = {
    "Crosses local midnight": (local_end.dt.date != local_begin.dt.date) & local_end.notna(),
    "Crosses UTC midnight": (events["end_utc"].dt.date != events["begin_utc"].dt.date)
    & events["end_utc"].notna(),
    "Crosses a UTC year boundary": events["end_utc"].dt.year > events["begin_utc"].dt.year,
    "Begins exactly on a six-hour UTC boundary": on_boundary,
    "Zero duration on a six-hour UTC boundary": on_boundary
    & (events["end_utc"] == events["begin_utc"]),
}
pd.DataFrame({"Events": {case: int(mask.sum()) for case, mask in CASES.items()}}).rename_axis(
    "Case"
)

### Open question: standard time or clock time?

Every North Carolina record says `EST-5`, including summer events. If NCEI's summer times
are actually clock time (EDT, UTC−4), then those events are **one hour late** in UTC. The
field text alone cannot settle this, so the size of the exposure is measured here and the
answer comes from comparing events against their NWS warning timestamps.

In [ ]:
in_dst = (
    events["begin_utc"].dt.tz_convert("America/New_York").map(lambda t: t.dst() != pd.Timedelta(0))
)
print(
    f"Events beginning while Eastern daylight time is in force: {int(in_dst.sum()):,} "
    f"of {len(events):,} ({pct(in_dst.sum(), len(events))})"
)

### Spot-check candidates

The gate requires at least ten events checked **by a person** against NOAA's event pages and
NWS products. These are chosen deterministically to cover the riskiest cases, with three
daylight-saving-period events to settle the question above. In full mode they are written to
`reports/eda/spot_check_candidates.csv`, with blank columns for the reviewer.

In [ ]:
durations = events["end_utc"] - events["begin_utc"]
helene = events["begin_utc"].between(
    pd.Timestamp("2024-09-25", tz="UTC"), pd.Timestamp("2024-09-29", tz="UTC")
)
strata = [
    ("Daylight-saving period (settles EST-5)", in_dst, 3),
    ("Standard-time period", ~in_dst, 1),
    ("Zero duration", durations == pd.Timedelta(0), 1),
    ("Crosses local midnight", CASES["Crosses local midnight"], 1),
    ("Debris Flow", events["event_type"] == "Debris Flow", 1),
    ("Helene period", helene, 1),
]
chosen: list[tuple[str, str]] = []
for label, mask, size in strata:
    pool = events[mask & ~events["event_id"].isin([e for e, _ in chosen])]
    chosen += [(e, label) for e in pool.sample(min(size, len(pool)), random_state=SEED)["event_id"]]
remaining = events[~events["event_id"].isin([e for e, _ in chosen])]
if len(remaining):
    chosen.append(
        (remaining.loc[durations[remaining.index].idxmax(), "event_id"], "Longest duration")
    )
remaining = events[~events["event_id"].isin([e for e, _ in chosen])]
fill = max(0, 10 - len(chosen))
chosen += [
    (e, "Random")
    for e in remaining.sample(min(fill, len(remaining)), random_state=SEED)["event_id"]
]

picked = events.set_index("event_id").loc[[e for e, _ in chosen]]
spot_check = pd.DataFrame(
    {
        "event_id": picked.index,
        "why_selected": [label for _, label in chosen],
        "event_type": picked["event_type"].astype(str).to_numpy(),
        "county_fips": picked["county_fips"].to_numpy(),
        "begin_as_published": published.loc[picked.index, "BEGIN_DATE_TIME"].to_numpy(),
        "begin_utc": picked["begin_utc"].dt.strftime("%Y-%m-%d %H:%M").to_numpy(),
        "ncei_event_page": [
            f"https://www.ncdc.noaa.gov/stormevents/eventdetails.jsp?id={e}" for e in picked.index
        ],
        "reviewer": "",
        "county_matches": "",
        "utc_time_matches_nws_product": "",
        "notes": "",
    }
)
if MODE == "full":
    target = REPO_ROOT / "reports" / "eda" / "spot_check_candidates.csv"
    spot_check.to_csv(target, index=False)
    print(f"written {target.relative_to(REPO_ROOT)}")
spot_check.iloc[:, :6]

In [ ]:
bad_end = report.excluded_invalid_end + report.excluded_end_before_begin
checked_ends = int(end_both.sum())
zero_on_boundary = int(CASES["Zero duration on a six-hour UTC boundary"].sum())
dst_events = int(in_dst.sum())
conclude(
    f"All {len(events):,} qualifying events have a valid begin time; "
    f"{n(bad_end, 'event')} had to be excluded for a bad end time. "
    "Ingestion's UTC timestamps agree with NCEI's own date-time text for "
    f"{len(events) - begin_mismatch:,} of {len(events):,} begin times and "
    f"{checked_ends - end_mismatch:,} of {checked_ends:,} end times. "
    f"{n(report.zero_duration, 'event')} ({pct(report.zero_duration, len(events))}) "
    f"have zero duration, and {zero_on_boundary:,} of them begin exactly on a window boundary: "
    "the onset rule places each in exactly one window. **Unresolved:** whether `EST-5` is "
    f"true for the {n(dst_events, 'daylight-saving-period event')} "
    f"({pct(dst_events, len(events))}). The spot check settles it."
)

## 5. Hazard and geography filters

**Question.** How many records survive each filter, what does the hazard filter leave out,
and can every event be placed in a county?

The filters run in `stormroute.data.noaa.clean_events`, which counts every excluded row.

In [ ]:
qualifying_rows = report.rows_qualifying_type
placeable = (
    qualifying_rows
    - report.exact_duplicates_removed
    - sum(report.excluded_unresolved_geography.values())
)
funnel = pd.DataFrame(
    [
        ("All states, all event types", len(raw)),
        (f"{DATA['geography']['state']} only", report.rows_in_state),
        ("Qualifying hazard types only", qualifying_rows),
        ("Exact duplicates removed", qualifying_rows - report.exact_duplicates_removed),
        ("County-coded (placeable in a county)", placeable),
        ("Valid timestamps: final events", report.events_out),
    ],
    columns=["Stage", "Records"],
).set_index("Stage")
funnel["Share of previous"] = [""] + [
    pct(b, a) for a, b in zip(funnel["Records"][:-1], funnel["Records"][1:], strict=True)
]
funnel

In [ ]:
qualifying_nc = raw_nc[
    raw_nc["EVENT_TYPE"].isin(DATA["label"]["hazard_event_types"])
].drop_duplicates()
geo = pd.DataFrame(
    [
        ("County-coded (CZ_TYPE = C)", int((qualifying_nc["CZ_TYPE"] == "C").sum())),
        ("Zone-coded (CZ_TYPE = Z)", int((qualifying_nc["CZ_TYPE"] == "Z").sum())),
        ("Has begin coordinates", int((qualifying_nc["BEGIN_LAT"] != "").sum())),
        ("Cannot be assigned to a county", sum(report.excluded_unresolved_geography.values())),
    ],
    columns=["Qualifying NC records", "Count"],
).set_index("Qualifying NC records")
geo

### What the hazard filter leaves out

The locked definition keeps exactly three event types. Several other water-related types
also appear in North Carolina. Grouping them as "water-related" is an analyst choice for
this display only; it does not change the filter.

In [ ]:
WATER_RELATED = [
    "Flash Flood",
    "Flood",
    "Debris Flow",
    "Heavy Rain",
    "Tropical Storm",
    "Coastal Flood",
    "Hurricane (Typhoon)",
    "Storm Surge/Tide",
    "Tropical Depression",
]
KEPT = DATA["label"]["hazard_event_types"]
DISCUSSED = {"Heavy Rain", "Coastal Flood"}  # excluded types the gate decision must address

nc_types = raw_nc.drop_duplicates()["EVENT_TYPE"].value_counts()
plot = pd.DataFrame(
    {"type": WATER_RELATED, "reports": [int(nc_types.get(t, 0)) for t in WATER_RELATED]}
)
plot = plot[plot["reports"] > 0].assign(kept=lambda d: d["type"].isin(KEPT))
plot = pd.concat(
    [
        plot[plot["kept"]].sort_values("reports", ascending=False),
        plot[~plot["kept"]].sort_values("reports", ascending=False),
    ]
)

fig, ax = plt.subplots(figsize=(10, 5.6))
y = np.arange(len(plot))[::-1]
ax.barh(
    y,
    plot["reports"],
    height=0.62,
    edgecolor=COLORS["surface"],
    linewidth=2,
    color=np.where(plot["kept"], COLORS["accent"], COLORS["context"]),
)
ax.set_yticks(y, plot["type"])
ax.tick_params(axis="y", length=0)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.set_xlim(0, plot["reports"].max() * 1.12)
ax.xaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))
too_short_to_read = plot["reports"].max() * 0.03
for yi, row in zip(y, plot.itertuples(), strict=True):
    if row.kept or row.type in DISCUSSED or row.reports < too_short_to_read:
        ax.text(
            row.reports + plot["reports"].max() * 0.012,
            yi,
            f"{row.reports:,}",
            va="center",
            color=COLORS["ink_secondary"],
            fontsize=10,
        )
ax.set_xlabel(f"Reports in North Carolina, {SOURCE_NOTE.split('files ')[-1]} (count)")
ax.set_title(
    f"The hazard filter keeps {int(plot['kept'].sum())} of {len(plot)} water-related event types",
    pad=28,
)
ax.text(
    0,
    1.02,
    "NOAA Storm Events reports by type; the label uses only the kept types",
    transform=ax.transAxes,
    color=COLORS["ink_secondary"],
    fontsize=11,
)
ax.legend(
    handles=[
        Patch(color=COLORS["accent"], label="Kept: qualifying hazard"),
        Patch(color=COLORS["context"], label="Excluded"),
    ],
    loc="lower right",
    frameon=False,
)
fig.text(
    0.01,
    -0.02,
    SOURCE_NOTE + ". Water-related grouping is for display only.",
    color=COLORS["muted"],
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "eda_event_type_counts")
display(fig)
plt.close(fig)

In [ ]:
excluded = plot[~plot["kept"]].set_index("type")["reports"]
unresolved = sum(report.excluded_unresolved_geography.values())
geography = (
    "is county-coded, so **no** event had to be dropped for unplaceable geography, and a "
    "zone-to-county crosswalk is unnecessary for this label."
    if unresolved == 0
    else f"except {n(unresolved, 'zone-coded record')} is county-coded; "
    "those are excluded and counted."
)
conclude(
    f"{report.events_out:,} of {len(raw):,} records survive ({pct(report.events_out, len(raw))}). "
    f"Every qualifying record {geography} "
    f"The filter leaves out {int(excluded.sum()):,} water-related reports, notably Heavy Rain "
    f"({int(excluded.get('Heavy Rain', 0)):,}) and Coastal Flood "
    f"({int(excluded.get('Coastal Flood', 0)):,}). Heavy Rain is excluded by ADR 0000. "
    "**The team still needs to confirm the Coastal Flood exclusion** at the gate."
)

## 6. Temporal coverage

**Question.** Is reporting stable across 2015–2024, or do trends reflect reporting rather than weather?

**This section must:**

- annual event counts (export `eda_events_by_year.png`)
- monthly seasonality (export `eda_monthly_seasonality.png`)
- event duration distribution
- reporting gaps or abrupt changes
- highlight the Hurricane Helene period without letting one storm stand in for the dataset

**Conclusion.** _TBD — state what the result shows, in one or two sentences, after the code runs._

<!-- TODO(milestone-2): add analysis cells below this heading. -->

## 7. Geographic distribution

**Question.** Where do reported events occur, and how much of that is county size or reporting bias?

**This section must:**

- events by county
- normalized event rate by county and year
- North Carolina choropleth (export `eda_county_choropleth.png`)
- counties with zero reported qualifying events
- discussion of reporting bias and county-size effects

**Conclusion.** _TBD — state what the result shows, in one or two sentences, after the code runs._

<!-- TODO(milestone-2): add analysis cells below this heading. -->

## 8. Descriptive impact analysis

**Question.** Why does this matter to a traveler — stated without sensationalizing harm?

**This section must:**

- injury, death, and property-damage distributions
- median and upper-tail damage by event type
- an explicit warning that these variables occur after the event and **cannot be predictors**
- one chart connecting the project to traveler safety

**Conclusion.** _TBD — state what the result shows, in one or two sentences, after the code runs._

<!-- TODO(milestone-2): add analysis cells below this heading. -->

## 9. Label construction experiment

**Question.** Can the locked label actually be built, how rare is the positive class, and
does every temporal split contain positives?

The grid is every North Carolina county crossed with every six-hour window from
2015-01-01 to 2024-12-31 UTC. A window is positive when a qualifying event **begins** in it
(`window_start <= begin < window_end`; ADR 0001). The specification's overlap rule is also
computed, so the cost of the choice is measured rather than assumed.

In [ ]:
from stormroute.data.validation import NC_COUNTY_FIPS

WINDOW = pd.Timedelta(hours=DATA["prediction_unit"]["window_hours"])
window_starts = pd.date_range(
    pd.Timestamp(f"{DATA['period']['start_year']}-01-01", tz="UTC"),
    pd.Timestamp(f"{DATA['period']['end_year'] + 1}-01-01", tz="UTC"),
    freq=WINDOW,
    inclusive="left",
)
grid = pd.MultiIndex.from_product(
    [sorted(NC_COUNTY_FIPS), window_starts], names=["county_fips", "window_start_utc"]
)
onset = pd.MultiIndex.from_arrays(
    [events["county_fips"].astype(str), events["begin_utc"].dt.floor(WINDOW)]
)
onset_in_grid = onset.isin(grid)
labels = pd.Series(grid.isin(onset), index=grid, name="label_flood_event")
print(f"Grid: {len(NC_COUNTY_FIPS)} counties x {len(window_starts):,} windows = {len(grid):,} rows")
print(f"Events whose onset falls outside the UTC grid: {int((~onset_in_grid).sum())}")

In [ ]:
# The specification's overlap rule, for comparison only.
first_window = events["begin_utc"].dt.floor(WINDOW)
span = (events["end_utc"].fillna(events["begin_utc"]).dt.ceil(WINDOW) - first_window) / WINDOW
exploded = events.loc[events.index.repeat(span.astype(int).clip(lower=1))]
exploded = exploded.assign(
    window_start_utc=exploded["begin_utc"].dt.floor(WINDOW)
    + WINDOW * exploded.groupby(level=0).cumcount()
)
exploded_end = exploded["end_utc"].fillna(exploded["begin_utc"])
overlapping = exploded[
    (exploded["begin_utc"] < exploded["window_start_utc"] + WINDOW)
    & (exploded_end > exploded["window_start_utc"])
]
overlap_labels = grid.isin(
    pd.MultiIndex.from_arrays(
        [overlapping["county_fips"].astype(str), overlapping["window_start_utc"]]
    )
)
events_lost_by_overlap = events.index.difference(overlapping.index).size

onset_positives = int(labels.sum())
overlap_positives = int(overlap_labels.sum())
pd.DataFrame(
    [
        ("County-windows in the grid", f"{len(grid):,}", f"{len(grid):,}"),
        ("Positive windows", f"{onset_positives:,}", f"{overlap_positives:,}"),
        ("Positive rate", pct(onset_positives, len(grid)), pct(overlap_positives, len(grid))),
        (
            "Negatives per positive",
            f"{(len(grid) - onset_positives) / max(onset_positives, 1):,.0f}",
            f"{(len(grid) - overlap_positives) / max(overlap_positives, 1):,.0f}",
        ),
        ("Events that label no window at all", "0", f"{events_lost_by_overlap:,}"),
    ],
    columns=["", "Onset rule (locked)", "Overlap rule (spec)"],
).set_index("")

### Why event rows alone cannot train a classifier

The events table holds **only positives**: every row is something that happened. A
classifier also needs the windows where nothing was reported, and those exist only in the
grid. They are the overwhelming majority of it, and a negative means *no reported event
began*, not *safe*.

In [ ]:
year_to_split = {
    year: name
    for name, spec in MODEL["splits"].items()
    if isinstance(spec, dict)
    for year in spec["years"]
}
split_order = list(dict.fromkeys(year_to_split[year] for year in sorted(year_to_split)))
frame = labels.reset_index()
frame["split"] = frame["window_start_utc"].dt.year.map(year_to_split)
by_split = (
    frame.groupby("split")["label_flood_event"]
    .agg(windows="size", positives="sum")
    .reindex(split_order)
)
by_split["positive rate"] = [
    pct(p, w) for p, w in zip(by_split["positives"], by_split["windows"], strict=True)
]
by_split["has positives"] = by_split["positives"] > 0
by_split

In [ ]:
county_names = (
    raw_nc[raw_nc["CZ_TYPE"] == "C"]
    .assign(fips=lambda d: "37" + d["CZ_FIPS"].str.zfill(3))
    .drop_duplicates("fips")
    .set_index("fips")["CZ_NAME"]
    .str.title()
)
by_month = frame.groupby(frame["window_start_utc"].dt.month)["label_flood_event"].mean() * 100
by_county = (
    frame.groupby("county_fips")["label_flood_event"].mean().sort_values(ascending=False) * 100
)
zero_counties = int((by_county == 0).sum())

fig, (ax_month, ax_county) = plt.subplots(1, 2, figsize=(12, 5.2), width_ratios=[1, 1.6])
ax_month.bar(
    by_month.index,
    by_month.to_numpy(),
    width=0.7,
    color=COLORS["accent"],
    edgecolor=COLORS["surface"],
    linewidth=2,
)
ax_month.set_xticks(
    range(1, 13),
    [
        m[:1]
        for m in (
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
        )
    ],
)
ax_month.set_title("By calendar month (UTC)", fontsize=12)
ax_month.set_ylabel("Positive county-windows (%)")

ax_county.bar(range(len(by_county)), by_county.to_numpy(), width=0.8, color=COLORS["accent"])
ax_county.set_xticks([])
ax_county.set_xlabel(
    f"{len(by_county)} counties, ranked"
    + (f" ({n(zero_counties, 'county', 'counties')} with no positives)" if zero_counties else "")
)
ax_county.set_title("By county", fontsize=12)
for rank, (fips, rate) in enumerate(by_county.head(3).items()):
    ax_county.annotate(
        county_names.get(fips, fips),
        (rank, rate),
        xytext=(8, -2),
        textcoords="offset points",
        fontsize=9,
        color=COLORS["ink_secondary"],
    )
for ax in (ax_month, ax_county):
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)

fig.suptitle(
    f"Only {pct(onset_positives, len(grid))} of county-windows are positive",
    x=0.01,
    y=0.99,
    ha="left",
    fontsize=14,
    fontweight="bold",
)
fig.text(
    0.01,
    0.925,
    "Share of six-hour windows in which a Flood, Flash Flood, or Debris Flow "
    "event began, 2015–2024",
    color=COLORS["ink_secondary"],
    fontsize=11,
)
fig.text(0.01, -0.02, SOURCE_NOTE + ". Onset rule, ADR 0001.", color=COLORS["muted"], fontsize=9)
fig.tight_layout(rect=(0, 0, 1, 0.92))
save_figure(fig, "eda_class_imbalance")
display(fig)
plt.close(fig)

In [ ]:
smallest_split = by_split["positives"].idxmin()
all_splits = bool(by_split["has positives"].all())
conclude(
    f"The label is constructible: {onset_positives:,} of {len(grid):,} county-windows are "
    f"positive ({pct(onset_positives, len(grid))}), about one per "
    f"{(len(grid) - onset_positives) / max(onset_positives, 1):,.0f} negatives. "
    + (
        f"Every split has positives; the smallest is {smallest_split} "
        f"({int(by_split.loc[smallest_split, 'positives']):,}). "
        if all_splits
        else "**At least one split has no positives, which fails the gate.** "
    )
    + f"The overlap rule would add {overlap_positives - onset_positives:,} windows of ongoing "
    f"events but give {n(events_lost_by_overlap, 'event')} no window at all. "
    + (
        f"{n(int((~onset_in_grid).sum()), 'event')} begin after 2024 in UTC and fall outside "
        "the grid. "
        if (~onset_in_grid).any()
        else "Every event falls inside the UTC grid. "
    )
    + "Caveat: December 31 evening events in the 2014 file, which is not downloaded, would begin "
    "on 1 January 2015 in UTC and belong in the grid. "
    "Imbalance this severe is why PR-AUC, not accuracy or ROC-AUC, is the headline metric."
)

## 10. Weather-feature feasibility check

**Question.** Are the weather features available and consistent, and can they be aligned to
event times without reading the future?

Hourly reanalysis comes from the [Open-Meteo archive API](https://open-meteo.com/en/docs/historical-weather-api)
for one **representative point** in a mountain, a Piedmont, and a coastal county. These are
city coordinates, not county centroids; centroids need the county boundaries (issue #3).
Full mode caches each response under `data/raw/weather/` (ignored). Sample mode reads a
48-hour tracked fixture.

In [ ]:
import httpx

WEATHER_POINTS = pd.DataFrame(
    [
        ("Mountain", "37021", "Buncombe", 35.5951, -82.5515),
        ("Piedmont", "37119", "Mecklenburg", 35.2271, -80.8431),
        ("Coastal", "37129", "New Hanover", 34.2257, -77.9447),
    ],
    columns=["region", "county_fips", "county", "latitude", "longitude"],
)
WEATHER_VARIABLES = [
    "precipitation",
    "temperature_2m",
    "wind_speed_10m",
    "wind_gusts_10m",
    "soil_moisture_0_to_7cm",
    "soil_moisture_28_to_100cm",
]
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_REQUESTS = {
    "era5_land": ("2024-01-01", "2024-12-31"),
    "era5_seamless": ("2014-12-29", f"{DATA['period']['end_year']}-12-31"),  # 72 h of lead-in
}


def load_open_meteo(model: str, start: str, end: str) -> tuple[pd.DataFrame, str]:
    cache = data_path("weather_raw") / "open_meteo_eda" / f"{model}_{start}_{end}.json"
    if not cache.exists():
        params = {
            "latitude": ",".join(map(str, WEATHER_POINTS["latitude"])),
            "longitude": ",".join(map(str, WEATHER_POINTS["longitude"])),
            "start_date": start,
            "end_date": end,
            "hourly": ",".join(WEATHER_VARIABLES),
            "models": model,
            "timezone": "GMT",
            "wind_speed_unit": "ms",
        }
        response = httpx.get(ARCHIVE_URL, params=params, timeout=300)
        response.raise_for_status()
        cache.parent.mkdir(parents=True, exist_ok=True)
        accessed = datetime.now(UTC).isoformat(timespec="seconds")
        cache.write_text(
            json.dumps({"accessed_at_utc": accessed, "params": params, "results": response.json()})
        )
    payload = json.loads(cache.read_text())
    frames = []
    for point, result in zip(
        WEATHER_POINTS.itertuples(index=False), payload["results"], strict=True
    ):
        hourly = pd.DataFrame(result["hourly"])
        hourly[WEATHER_VARIABLES] = hourly[WEATHER_VARIABLES].astype("float64")  # None -> NaN
        hourly.insert(0, "time_utc", pd.to_datetime(hourly.pop("time")).dt.tz_localize("UTC"))
        frames.append(
            hourly.assign(
                model=model,
                region=point.region,
                county_fips=point.county_fips,
                grid_cell=f"{result['latitude']:.2f}, {result['longitude']:.2f}",
            )
        )
    return pd.concat(frames, ignore_index=True), payload["accessed_at_utc"]


if MODE == "sample":
    weather = pd.read_csv(data_path("sample") / "weather_sample.csv", dtype={"county_fips": str})
    weather["time_utc"] = pd.to_datetime(weather["time_utc"], utc=True)
    print("Sample mode: tracked 48-hour fixture, no network access")
else:
    loaded = {model: load_open_meteo(model, *span) for model, span in WEATHER_REQUESTS.items()}
    weather = pd.concat([frame for frame, _ in loaded.values()], ignore_index=True)
    for model, (_, accessed) in loaded.items():
        print(f"{model:14s} {' to '.join(WEATHER_REQUESTS[model])}, accessed {accessed}")
weather.pivot_table(index="region", columns="model", values="grid_cell", aggfunc="first")

In [ ]:
availability = (
    weather.groupby("model")[WEATHER_VARIABLES]
    .agg(lambda s: pct(s.notna().sum(), len(s)))
    .T.rename_axis("Variable (non-missing hours)")
)
availability

In [ ]:
# Where does each era5_seamless variable come from? Compare it with era5_land hour by hour.
# dropna=False keeps era5_land's all-missing columns, which are exactly what is being checked.
both = weather.pivot_table(
    index=["region", "time_utc"], columns="model", values=WEATHER_VARIABLES, dropna=False
)
fetched_both = (
    both[("temperature_2m", "era5_land")].notna()
    & both[("temperature_2m", "era5_seamless")].notna()
)
compared = both[fetched_both]
matches_land = {
    variable: pct(
        (compared[(variable, "era5_seamless")] == compared[(variable, "era5_land")]).sum(),
        len(compared),
    )
    for variable in WEATHER_VARIABLES
}
print(f"Compared over the {len(compared):,} point-hours fetched from both models")
pd.Series(matches_land, name="era5_seamless hours identical to era5_land").to_frame()

### Leakage-safe rolling features

Open-Meteo reports each hour's precipitation as the total over the **preceding** hour, so
the value stamped at time T is already known at T. The feature for a window starting at T
sums the last *k* stamped values ending at T. It never touches an hour after T, and the
perturbation test below proves it.

In [ ]:
ROLLING_HOURS = DATA["weather"]["rolling_precip_hours"]


def trailing_features(precipitation: pd.Series) -> pd.DataFrame:
    features = {
        f"precip_{k}h": precipitation.rolling(k, min_periods=k).sum() for k in ROLLING_HOURS
    }
    features["max_precip_intensity"] = precipitation.rolling(6, min_periods=6).max()
    return pd.DataFrame(features)


seamless = weather[weather["model"] == "era5_seamless"].sort_values(["county_fips", "time_utc"])
gaps = seamless.groupby("county_fips")["time_utc"].diff().dropna()
assert (gaps == pd.Timedelta(hours=1)).all(), "hourly series has gaps; rolling sums would be wrong"

features = pd.concat(
    [
        trailing_features(group.set_index("time_utc")["precipitation"]).assign(county_fips=fips)
        for fips, group in seamless.groupby("county_fips")
    ]
).reset_index()

# Perturbation test: changing every hour after T must leave the features at T unchanged.
probe = seamless[seamless["county_fips"] == "37021"].set_index("time_utc")["precipitation"]
probe_time = probe.index[len(probe) // 2].floor(WINDOW)
perturbed = probe.where(probe.index <= probe_time, probe + 1000.0)
unchanged = (
    trailing_features(probe).loc[probe_time].equals(trailing_features(perturbed).loc[probe_time])
)
assert unchanged, "a rolling feature read an hour after the window start"
print(f"Perturbation test at {probe_time:%Y-%m-%d %H:%M} UTC: features unchanged by future hours")

In [ ]:
at_window_start = features[features["time_utc"].dt.hour % 6 == 0].rename(
    columns={"time_utc": "window_start_utc"}
)
aligned = labels.reset_index().merge(at_window_start, on=["county_fips", "window_start_utc"])
negative_p95 = aligned.loc[~aligned["label_flood_event"], "precip_24h"].quantile(0.95)
alignment = (
    aligned.groupby("label_flood_event")
    .agg(
        windows=("precip_24h", "size"),
        median_precip_24h_mm=("precip_24h", "median"),
        median_precip_72h_mm=("precip_72h", "median"),
    )
    .rename(index={True: "Positive (event began)", False: "Negative"})
)
above_p95 = aligned.loc[aligned["label_flood_event"], "precip_24h"].gt(negative_p95).mean()
print(
    f"Positive windows whose trailing 24 h rain exceeds the negatives' 95th percentile "
    f"({negative_p95:.1f} mm): {pct(above_p95, 1)}"
)
alignment.round(1)

In [ ]:
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

HELENE = (pd.Timestamp("2024-09-24", tz="UTC"), pd.Timestamp("2024-09-30", tz="UTC"))
start = max(HELENE[0], seamless["time_utc"].min())
stop = min(HELENE[1], seamless["time_utc"].max())
shown = seamless[seamless["time_utc"].between(start, stop)]
ymax = max(shown["precipitation"].max(), 1.0)

fig, axes = plt.subplots(len(WEATHER_POINTS), 1, figsize=(11, 7), sharex=True, sharey=True)
for ax, point in zip(axes, WEATHER_POINTS.itertuples(index=False), strict=True):
    rain = shown[shown["county_fips"] == point.county_fips]
    ax.bar(
        rain["time_utc"],
        rain["precipitation"],
        width=0.9 / 24,  # days
        color=COLORS["accent"],
    )
    onsets = events.loc[
        (events["county_fips"] == point.county_fips) & events["begin_utc"].between(start, stop),
        "begin_utc",
    ]
    ax.plot(
        onsets,
        [ymax * 1.12] * len(onsets),
        marker="v",
        markersize=9,
        linestyle="none",
        color="#eb6834",
        markeredgecolor=COLORS["surface"],
    )
    ax.set_ylim(0, ymax * 1.25)
    ax.set_title(
        f"{point.region}: {point.county} County ({n(len(onsets), 'reported onset')})",
        fontsize=11,
        loc="left",
    )
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
axes[len(axes) // 2].set_ylabel("Precipitation (mm per hour)")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[-1].set_xlabel("Time (UTC)")
axes[0].legend(
    handles=[
        Patch(color=COLORS["accent"], label="Hourly precipitation"),
        Line2D(
            [],
            [],
            marker="v",
            linestyle="none",
            color="#eb6834",
            markersize=9,
            label="Reported flood onset (NOAA)",
        ),
    ],
    loc="upper right",
    frameon=False,
)
fig.suptitle(
    "Hourly rainfall and reported flood onsets during Hurricane Helene",
    x=0.01,
    y=0.99,
    ha="left",
    fontsize=14,
    fontweight="bold",
)
fig.text(
    0.01,
    0.945,
    "ERA5 precipitation via Open-Meteo (era5_seamless), one representative point per county",
    color=COLORS["ink_secondary"],
    fontsize=11,
)
fig.text(
    0.01,
    -0.01,
    "Sources: Open-Meteo archive API (ERA5, Copernicus C3S); " + SOURCE_NOTE[8:] + ".",
    color=COLORS["muted"],
    fontsize=9,
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
save_figure(fig, "eda_precipitation_event_comparison")
display(fig)
plt.close(fig)

In [ ]:
land_precip = availability.loc["precipitation", "era5_land"]
median_24h = alignment["median_precip_24h_mm"]


def mm(label: str) -> str:
    value = median_24h.get(label, float("nan"))
    return "n/a" if pd.isna(value) else f"{value:.1f} mm"


conclude(
    "Weather features are available but **not from the configured source as-is**. Through "
    f"Open-Meteo, ERA5-Land has precipitation and wind for {land_precip} of hours, so they are "
    "missing entirely. The `era5_seamless` model fills them from ERA5, a coarser (~28 km) grid, "
    "while temperature and soil moisture stay on ERA5-Land's ~9 km grid. Every variable is present "
    "in `era5_seamless` for the hours fetched. Rolling features pass the perturbation test, "
    "so they "
    "read nothing after the window start. At window start, positive windows have a median trailing "
    f"24 h rainfall of {mm('Positive (event began)')} versus {mm('Negative')} for negatives; "
    "this is an alignment check, "
    "not model evidence. **Gate decision needed:** accept ERA5 rainfall via Open-Meteo, or fetch "
    "ERA5-Land precipitation directly from Copernicus (account and API key required)."
)

## 11. Leakage audit

**Question.** Could any candidate feature carry information from after departure?

Each candidate is classified as **known before departure**, **contemporaneous** (true during
the window, so unknown at departure), or **known only after the event**. The table is then
checked against `configs/model.yaml`, so the configuration cannot quietly disagree with it.

In [ ]:
BEFORE, DURING, AFTER = "Known before departure", "Contemporaneous", "Known only after the event"
AUDIT = [
    (
        ["precip_1h", "precip_3h", "precip_6h", "precip_24h", "precip_72h"],
        "Reanalysis",
        BEFORE,
        "Keep",
        "Trailing sums ending at window start; perturbation test passes (section 10)",
    ),
    (
        ["max_precip_intensity"],
        "Reanalysis",
        BEFORE,
        "Keep",
        "Only as a trailing 6 h maximum; a maximum over the window itself would read the future",
    ),
    (
        ["temperature", "wind_speed", "wind_gust"],
        "Reanalysis",
        BEFORE,
        "Keep",
        "Value at window start",
    ),
    (
        ["soil_moisture_shallow", "soil_moisture_deep"],
        "Reanalysis",
        BEFORE,
        "Keep",
        "Value at window start; antecedent wetness",
    ),
    (["elevation_mean", "slope_mean"], "Terrain", BEFORE, "Keep", "Static"),
    (["month_sin", "month_cos"], "Calendar", BEFORE, "Keep", "Known in advance"),
    (
        ["historical_event_rate"],
        "NOAA labels",
        BEFORE,
        "Conditional",
        "Only if fit on 2015–2021; may encode reporting bias, so evaluated with and without",
    ),
    (
        ["rain during the window"],
        "Reanalysis",
        DURING,
        "Reject",
        "Unknown at departure; a live system would need a forecast (ADR 0003)",
    ),
    (
        ["centred or whole-period weather aggregates"],
        "Reanalysis",
        AFTER,
        "Reject",
        "Averages over hours after the window start",
    ),
    (["injuries", "deaths"], "NOAA", AFTER, "Reject", "Outcomes recorded after the event"),
    (["property_damage_usd", "crop damage"], "NOAA", AFTER, "Reject", "Assessed after the event"),
    (["event_narrative", "episode narrative"], "NOAA", AFTER, "Reject", "Written after the event"),
    (["end_utc", "event duration"], "NOAA", AFTER, "Reject", "Unknown until the event ends"),
    (["episode_id"], "NOAA", AFTER, "Reject", "Assigned during post-event curation"),
    (
        ["magnitude, flood cause, reporting source"],
        "NOAA",
        AFTER,
        "Reject",
        "Exist only for reported events, so they would identify positives perfectly",
    ),
    (
        ["other NOAA reports in the same window (e.g. Heavy Rain)"],
        "NOAA",
        DURING,
        "Reject",
        "Filed during or after the window",
    ),
    (
        ["Social Vulnerability Index"],
        "CDC/ATSDR",
        BEFORE,
        "Reject",
        "Policy: must never change an individual trip score (responsible AI)",
    ),
]
audit = pd.DataFrame(AUDIT, columns=["Features", "Source", "Availability", "Decision", "Reason"])
audit.assign(Features=audit["Features"].map(", ".join)).set_index("Features")

In [ ]:
decision = {name: row.Decision for row in audit.itertuples() for name in row.Features}
configured = [f for group in ("weather", "static", "temporal") for f in MODEL["features"][group]]
optional = MODEL["features"]["optional"]
excluded_by_config = list(MODEL["features"]["excluded"])

problems = (
    [
        f"{f}: configured as a feature but not audited as Keep"
        for f in configured
        if decision.get(f) != "Keep"
    ]
    + [
        f"{f}: optional feature not audited as Conditional"
        for f in optional
        if decision.get(f) != "Conditional"
    ]
    + [
        f"{f}: excluded in config but audited as {decision.get(f)}"
        for f in excluded_by_config
        if f in decision and decision[f] != "Reject"
    ]
)
unaudited_exclusions = [f for f in excluded_by_config if f not in decision]
print(f"Configured features: {len(configured)} kept, {len(optional)} optional")
print("Config agrees with the audit" if not problems else "\n".join(problems))
if unaudited_exclusions:
    print(f"Excluded in config under other names (covered above in words): {unaudited_exclusions}")

In [ ]:
counts = audit["Decision"].value_counts()
conclude(
    f"{n(len(audit), 'candidate group')} audited: {int(counts.get('Keep', 0))} kept, "
    f"{int(counts.get('Conditional', 0))} conditional, {int(counts.get('Reject', 0))} rejected. "
    + (
        "The configured feature list agrees with the audit. "
        if not problems
        else f"**{n(len(problems), 'disagreement')} between config and audit.** "
    )
    + "Every kept feature is known at the window start. The remaining risk is not leakage but "
    "**train-serve skew**: features are built from reanalysis, which a live deployment would not "
    "have, so evaluated performance is an upper bound on live performance (ADR 0003)."
)

## 12. EDA conclusions and model decisions

The gate decision. Each answer must be backed by a number or a figure above.

| # | Question | Answer | Evidence |
|---|---|---|---|
| 1 | Is the proposed label constructible? | _TBD_ | |
| 2 | Is the positive class large enough for the proposed temporal splits? | _TBD_ | |
| 3 | Which features are available consistently? | _TBD_ | |
| 4 | Which fields are excluded, and why? | _TBD_ | |
| 5 | Which data-quality risks remain? | _TBD_ | |
| 6 | **Does the team proceed, change scope, or stop?** | _TBD_ | |

**Decision recorded by:** _TBD (Mason)_ · **Reviewed by:** _TBD (Econ/Stats)_, _TBD (CS)_

<!-- TODO(milestone-2): complete after the full-data run. -->